In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
import os

# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 10
DATA_DIR = './PlantVillage'

# The three crops isolated for the Model Router
CROPS = ['tomato', 'potato', 'pepper']

# Standard ImageNet transformations
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
def train_crop_model(crop_name):
    print(f"\n--- Initiating Transfer Learning for {crop_name.upper()} ---")
    crop_dir = os.path.join(DATA_DIR, crop_name)
    
    # 1. Load Isolated Dataset
    dataset = datasets.ImageFolder(crop_dir, transform=data_transforms)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    num_classes = len(dataset.classes)
    print(f"Classes detected for {crop_name}: {dataset.classes}")

    # 2. Initialize Pre-trained ResNet-18
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # Freeze core convolutional layers to retain ImageNet feature extraction
    for param in model.parameters():
        param.requires_grad = False
        
    # Modify the final fully connected (fc) layer for the specific crop classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # 3. Define Loss Function and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

    # 4. Training Loop
    model.train()
    for epoch in range(EPOCHS):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        epoch_loss = running_loss / len(dataset)
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f}")

    # 5. Save isolated weights for the Streamlit Model Router
    save_path = f'agrinexus_{crop_name}_resnet18.pth'
    torch.save(model.state_dict(), save_path)
    print(f"Model successfully isolated and saved to: {save_path}")

In [ ]:
# Execute sequential training for all router models
for crop in CROPS:
    train_crop_model(crop)